<a href="https://colab.research.google.com/github/fernandesladeira/GerQueijo/blob/main/ApplosChallenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# generate_data.py
from faker import Faker
import csv
import random
from datetime import datetime, timedelta
import pandas as pd
import json

fake = Faker()

# 1. GERAR DADOS
print("Gerando customers.csv...")
with open('customers.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['id', 'name', 'age', 'region'])
    for i in range(1, 101):
        writer.writerow([i, fake.name(), random.randint(18, 70), fake.state()])

print("Gerando products.csv...")
categories = ['Electronics', 'Fashion', 'Home', 'Sports']
with open('products.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['id', 'category', 'price', 'supplier'])
    for i in range(1, 51):
        writer.writerow([i, random.choice(categories), round(random.uniform(10, 1000), 2), fake.company()])

print("Gerando sales.csv...")
with open('sales.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['id', 'customer_id', 'product_id', 'date', 'quantity'])
    for i in range(1, 501):
        writer.writerow([
            i,
            random.randint(1, 100),
            random.randint(1, 50),
            fake.date_between(start_date='-1y', end_date='today'),
            random.randint(1, 5)
        ])

# 2. FUNÇÕES DE LIMPEZA
def clean_customers(df):
    df.drop_duplicates(subset='id', inplace=True)
    df['age'] = df['age'].fillna(df['age'].median())
    return df

def clean_sales(df):
    df = df.dropna(subset=['customer_id', 'product_id', 'quantity'])
    df = df[df['quantity'] > 0]
    # Tratar outliers
    Q1 = df['quantity'].quantile(0.25)
    Q3 = df['quantity'].quantile(0.75)
    IQR = Q3 - Q1
    df = df[df['quantity'] <= Q3 + 1.5 * IQR]
    df['date'] = pd.to_datetime(df['date'])
    return df

def generate_metrics(sales_df, products_df, customers_df):
    # Padronizar nomes das colunas de ID para facilitar os merges
    products_df = products_df.rename(columns={'id': 'product_id'})
    customers_df = customers_df.rename(columns={'id': 'customer_id'})

    # Realizar os merges
    merged = sales_df.merge(products_df, on='product_id', how='inner')
    merged = merged.merge(customers_df, on='customer_id', how='inner')

    # Calcular métricas
    revenue_by_region = merged.groupby('region')['price'].sum().to_dict()

    top_products_series = merged.groupby('product_id')['quantity'].sum().nlargest(5)
    top_products = {str(k): int(v) for k, v in top_products_series.to_dict().items()}

    churn_candidates = merged.groupby('customer_id')['date'].max().apply(
        lambda x: x < pd.Timestamp.now() - pd.Timedelta(days=180)
    ).sum()

    return {
        'revenue_by_region': revenue_by_region,
        'top_products': top_products,
        'churn_candidates': int(churn_candidates)
    }

# 3. EXECUTAR PIPELINE
print("\nIniciando pipeline ETL...")

# Ler CSVs
customers_df = pd.read_csv('customers.csv')
products_df = pd.read_csv('products.csv')
sales_df = pd.read_csv('sales.csv')

print(f"Customers: {len(customers_df)} registros")
print(f"Products: {len(products_df)} registros")
print(f"Sales: {len(sales_df)} registros")

# Aplicar limpeza
customers_clean = clean_customers(customers_df)
sales_clean = clean_sales(sales_df)
products_clean = products_df  # products já está limpo

print(f"Após limpeza - Sales: {len(sales_clean)} registros")

# Gerar métricas
metrics = generate_metrics(sales_clean, products_clean, customers_clean)

# Salvar resultado
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("\n✅ Pipeline concluído!")
print("📊 Métricas salvas em metrics.json")
print("\nResumo das métricas:")
print(f"- Regiões com vendas: {len(metrics['revenue_by_region'])}")
print(f"- Top 5 produtos: {list(metrics['top_products'].keys())}")
print(f"- Clientes em churn: {metrics['churn_candidates']}")

# 4. MOSTRAR AMOSTRA DOS DADOS (opcional)
print("\n--- Amostra das métricas geradas ---")
print("Revenue por região (top 3):")
for i, (region, revenue) in enumerate(sorted(metrics['revenue_by_region'].items(), key=lambda x: x[1], reverse=True)[:3]):
    print(f"  {i+1}. {region}: R${revenue:.2f}")

print("\nTop 5 produtos:")
for product_id, quantity in metrics['top_products'].items():
    print(f"  Produto {product_id}: {quantity} unidades")

Gerando customers.csv...
Gerando products.csv...
Gerando sales.csv...

Iniciando pipeline ETL...
Customers: 100 registros
Products: 50 registros
Sales: 500 registros
Após limpeza - Sales: 500 registros

✅ Pipeline concluído!
📊 Métricas salvas em metrics.json

Resumo das métricas:
- Regiões com vendas: 40
- Top 5 produtos: ['34', '30', '26', '32', '2']
- Clientes em churn: 8

--- Amostra das métricas geradas ---
Revenue por região (top 3):
  1. Indiana: R$17971.85
  2. Connecticut: R$15810.30
  3. New York: R$12500.98

Top 5 produtos:
  Produto 34: 59 unidades
  Produto 30: 46 unidades
  Produto 26: 44 unidades
  Produto 32: 43 unidades
  Produto 2: 42 unidades


In [ ]:
from google.colab import drive
drive.mount('/content/drive')